In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, roc_auc_score 

# ==========================================
# 1. 基础配置与加载
# ==========================================
def load_data():
    try:
        df1 = pd.read_csv('D:/R-work/1-age-gender.csv')
        df2 = pd.read_csv('D:/R-work/2-age-gender.csv')
        df3 = pd.read_csv('D:/R-work/3-age-gender.csv')
        return df1, df2, df3
    except Exception as e:
        print(f"数据加载失败: {e}")
        return None, None, None

def get_raw_group(pid):
    if not isinstance(pid, str): return None
    parts = pid.split('.')
    for p in parts:
        if p == 'G': return 'G'
        if p == 'SE': return 'NM' # SE -> NM
        if p == 'SL': return 'NB' # SL -> NB
    return None

def process_batch_subset_normalize(df, group1, group2, protein_cols):

    d = df.copy()
    d['Group'] = d['Protein.Ids'].apply(get_raw_group)
    
    # 提取两组
    d = d[d['Group'].isin([group1, group2])].copy()
    
    if len(d) == 0:
        return d
    
    # 仅针对该子集标准化
    cols_to_scale = [c for c in protein_cols if c in d.columns]
    
    if cols_to_scale:
        scaler = StandardScaler()
        d[cols_to_scale] = scaler.fit_transform(d[cols_to_scale])
        
    return d


def calculate_auc_ci(y_true, y_pred_prob, n_bootstraps=1000, alpha=0.05, random_state=3):
    rng = np.random.RandomState(random_state)
    bootstrapped_scores = []
    
    y_true = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred_prob), len(y_pred_prob))
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        score = roc_auc_score(y_true[indices], y_pred_prob[indices])
        bootstrapped_scores.append(score)
        
    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    

    lower_bound = np.percentile(sorted_scores, (alpha / 2) * 100)
    upper_bound = np.percentile(sorted_scores, (1.0 - alpha / 2) * 100)
    
    return lower_bound, upper_bound


df1, df2, df3 = load_data()


common_cols = set(df1.columns).intersection(df2.columns).intersection(df3.columns)
meta_cols = ['Protein.Ids', 'Age', 'Gender']
protein_cols = [c for c in common_cols if c not in meta_cols and 'LOG2' not in c]
protein_cols.sort()

comparisons = [
    ('G', 'NM', 'G vs NM'),
    ('G', 'NB', 'G vs NB'),
    ('NM', 'NB', 'NM vs NB')
]


gender_order = [(1, 'Male'), (0, 'Female')]


fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for i, (g_code, g_name) in enumerate(gender_order):
    ax = axes[i]
    
    for g1, g2, label in comparisons:

        df1_sub = process_batch_subset_normalize(df1, g1, g2, protein_cols)
        df2_sub = process_batch_subset_normalize(df2, g1, g2, protein_cols)
        df3_sub = process_batch_subset_normalize(df3, g1, g2, protein_cols)
        

        train_df = pd.concat([df1_sub, df2_sub], ignore_index=True)
        test_df = df3_sub
        

        tr = train_df[train_df['Gender'] == g_code].copy()
        te = test_df[test_df['Gender'] == g_code].copy()
        

        if len(tr) < 5 or len(te) < 3:
            print(f"  {label}: 样本量不足，跳过")
            continue
            

        tr['Target'] = tr['Group'].apply(lambda x: 1 if x == g1 else 0)
        te['Target'] = te['Group'].apply(lambda x: 1 if x == g1 else 0)
        
        y_train = tr['Target'].astype(int)
        y_test = te['Target'].astype(int)
        
        X_train = tr[protein_cols].fillna(0)
        X_test = te[protein_cols].fillna(0)
        

        selector = SelectKBest(f_classif, k=6)
        selector.fit(X_train, y_train)
        
        X_train_sel = selector.transform(X_train)
        X_test_sel = selector.transform(X_test)
        

        clf = LogisticRegression(max_iter=5000, random_state=3)
        clf.fit(X_train_sel, y_train)
        

        y_prob = clf.predict_proba(X_test_sel)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = auc(fpr, tpr)

        ci_lower, ci_upper = calculate_auc_ci(y_test, y_prob, n_bootstraps=1000)
        

        legend_text = f'{label} (AUC = {auc_val:.2f}, 95% CI: {ci_lower:.2f}-{ci_upper:.2f})'
        ax.plot(fpr, tpr, lw=6, label=legend_text)


    ax.plot([0, 1], [0, 1], 'k--', lw=3, alpha=0.5)
    ax.set_xlim([-0.02, 1.0])
    ax.set_ylim([0.0, 1.05])
    
    ax.set_xlabel('False Positive Rate', fontsize=24, fontweight='bold', labelpad=10)
    ax.set_ylabel('True Positive Rate', fontsize=24, fontweight='bold', labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.set_title(f'{g_name} Group', fontsize=24, fontweight='bold', pad=15)
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontweight('bold')
        
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_color('black')
    ax.legend(loc="lower right", fontsize=13, frameon=True)
    ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('roc_final_with_ci.png', dpi=600, bbox_inches='tight')
plt.show()
